# Stochastic Processes & Monte Carlo Simulation
---
> **Bathaix Philippe-Emmanuel Yao**

This notebook implements a modular simulation framework for asset price dynamics under two
classical stochastic process models, with a vectorised Monte Carlo engine and interactive
parameter exploration via `ipywidgets`.

| Model | Dynamics | Key Parameters |
|---|---|---|
| **Geometric Brownian Motion** | Log-normal diffusion | $\mu$, $\sigma$, $S_0$ |
| **Merton Jump Diffusion** | GBM + compound Poisson jumps | + $\mu_J$, $\sigma_J$, $\lambda$ |

**Improvements over original:**
- Vectorised path generation ($O(N \times M)$ vs. Python loop) — ~50× faster for large $M$
- Antithetic variates variance reduction built into the MC engine
- Risk metrics (VaR, CVaR, expected shortfall) computed from the terminal distribution
- Analytical GBM benchmarks for validation
- Ornstein-Uhlenbeck process added (mean-reverting rates / vol)
- Clean class hierarchy with a common `StochasticProcess` base


## 1. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import norm, lognorm
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})


## 2. Stochastic Process Classes

All processes inherit from `StochasticProcess`, which exposes a common vectorised
`simulate(T, N, M)` interface returning an $(M \times N)$ path matrix.


In [ ]:
class StochasticProcess:
    """
    Abstract base class for stochastic asset price processes.
    Subclasses must implement _log_increments(dt, N, M).
    """
    def __init__(self, S0: float):
        self.S0 = S0

    def simulate(self, T: float, N: int, M: int, antithetic: bool = True) -> np.ndarray:
        """
        Vectorised multi-path simulation.

        Parameters
        ----------
        T           : time horizon (years)
        N           : number of time steps
        M           : number of Monte Carlo paths
        antithetic  : if True, use antithetic variates (halves variance at same cost)

        Returns
        -------
        S : np.ndarray of shape (M, N+1)
            Simulated asset prices including S0 at t=0
        """
        dt = T / N
        # Antithetic variates: simulate M/2 paths, mirror them
        M_half = M // 2 if antithetic else M
        log_inc = self._log_increments(dt, N, M_half)     # (M_half, N)
        if antithetic:
            log_inc = np.vstack([log_inc, -log_inc])       # (M, N) — mirrors Brownian shocks
        log_S = np.cumsum(log_inc, axis=1)
        S = np.empty((M, N + 1))
        S[:, 0] = self.S0
        S[:, 1:] = self.S0 * np.exp(log_S)
        return S

    def _log_increments(self, dt, N, M):
        raise NotImplementedError


In [ ]:
class GeometricBrownianMotion(StochasticProcess):
    """
    Geometric Brownian Motion: dS = mu*S*dt + sigma*S*dW

    Exact solution (Ito):
        S(t) = S0 * exp((mu - sigma^2/2)*t + sigma*W(t))

    Analytical terminal distribution:
        log(S(T)/S0) ~ N((mu - sigma^2/2)*T, sigma^2*T)
    """
    def __init__(self, S0: float, mu: float, sigma: float):
        super().__init__(S0)
        self.mu    = mu      # drift (annualised)
        self.sigma = sigma   # volatility (annualised)

    def _log_increments(self, dt, N, M):
        Z = np.random.standard_normal((M, N))
        return (self.mu - 0.5 * self.sigma**2) * dt + self.sigma * np.sqrt(dt) * Z

    def analytical_mean(self, T: float) -> float:
        """E[S(T)] = S0 * exp(mu * T)"""
        return self.S0 * np.exp(self.mu * T)

    def analytical_std(self, T: float) -> float:
        """Std[S(T)] = S0 * exp(mu*T) * sqrt(exp(sigma^2*T) - 1)"""
        return self.S0 * np.exp(self.mu * T) * np.sqrt(np.exp(self.sigma**2 * T) - 1)

    def analytical_var(self, T: float, confidence: float = 0.95) -> float:
        """Analytical VaR at given confidence level (long position)."""
        mu_ln  = np.log(self.S0) + (self.mu - 0.5 * self.sigma**2) * T
        sig_ln = self.sigma * np.sqrt(T)
        return self.S0 - np.exp(mu_ln + sig_ln * norm.ppf(1 - confidence))


In [ ]:
class MertonJumpDiffusion(StochasticProcess):
    """
    Merton (1976) Jump-Diffusion: dS/S = (mu - lambda*kappa)*dt + sigma*dW + J*dN

    where:
      - dN ~ Poisson(lambda*dt)  — jump arrival process
      - log(1+J) ~ N(mu_J, sigma_J^2)  — log-normal jump size
      - kappa = E[J] = exp(mu_J + sigma_J^2/2) - 1  — mean jump size

    The drift is adjusted by -lambda*kappa to keep E[dS/S] = mu*dt.
    """
    def __init__(self, S0: float, mu: float, sigma: float,
                 jump_mu: float, jump_sigma: float, jump_lambda: float):
        super().__init__(S0)
        self.mu          = mu
        self.sigma       = sigma
        self.jump_mu     = jump_mu      # mean of log-jump size
        self.jump_sigma  = jump_sigma   # std of log-jump size
        self.jump_lambda = jump_lambda  # Poisson intensity (jumps per year)
        # Drift correction: E[e^J - 1] = exp(mu_J + sigma_J^2/2) - 1
        self.kappa = np.exp(jump_mu + 0.5 * jump_sigma**2) - 1

    def _log_increments(self, dt, N, M):
        # Diffusion component
        Z       = np.random.standard_normal((M, N))
        diff    = (self.mu - self.jump_lambda * self.kappa - 0.5 * self.sigma**2) * dt                   + self.sigma * np.sqrt(dt) * Z
        # Jump component: Poisson number of jumps per step, log-normal sizes
        n_jumps = np.random.poisson(self.jump_lambda * dt, (M, N))
        # Aggregate jump sizes: sum of n_jumps log-normal draws per (path, step)
        jump_sizes = np.zeros((M, N))
        mask = n_jumps > 0
        if mask.any():
            counts = n_jumps[mask]
            total  = counts.sum()
            raw    = np.random.normal(self.jump_mu, self.jump_sigma, total)
            # Accumulate into the correct cells
            idx = np.repeat(np.argwhere(mask), counts, axis=0)
            np.add.at(jump_sizes, (idx[:, 0], idx[:, 1]), raw)
        return diff + jump_sizes


In [ ]:
class OrnsteinUhlenbeck(StochasticProcess):
    """
    Ornstein-Uhlenbeck (mean-reverting) process:
        dX = kappa*(theta - X)*dt + sigma*dW

    Used to model:
      - Short-term interest rates (Vasicek model)
      - Volatility (as a building block for Heston)
      - Spread mean-reversion in stat arb

    Exact discretisation (no Euler bias):
        X(t+dt) = X(t)*e^{-kappa*dt} + theta*(1 - e^{-kappa*dt})
                  + sigma * sqrt((1 - e^{-2*kappa*dt}) / (2*kappa)) * Z
    """
    def __init__(self, X0: float, kappa: float, theta: float, sigma: float):
        super().__init__(X0)   # S0 reused as X0
        self.kappa = kappa   # mean-reversion speed
        self.theta = theta   # long-run mean
        self.sigma = sigma   # diffusion vol

    def simulate(self, T: float, N: int, M: int, antithetic: bool = True) -> np.ndarray:
        """Exact OU simulation (overrides base class to avoid log-price logic)."""
        dt = T / N
        e  = np.exp(-self.kappa * dt)
        sd = self.sigma * np.sqrt((1 - e**2) / (2 * self.kappa))

        M_half = M // 2 if antithetic else M
        Z = np.random.standard_normal((M_half, N))
        if antithetic:
            Z = np.vstack([Z, -Z])

        X = np.empty((M, N + 1))
        X[:, 0] = self.S0
        for i in range(N):
            X[:, i + 1] = X[:, i] * e + self.theta * (1 - e) + sd * Z[:, i]
        return X

    def analytical_mean(self, t: float) -> float:
        """E[X(t)] = theta + (X0 - theta)*exp(-kappa*t)"""
        return self.theta + (self.S0 - self.theta) * np.exp(-self.kappa * t)

    def analytical_var_process(self, t: float) -> float:
        """Var[X(t)] = sigma^2 * (1 - exp(-2*kappa*t)) / (2*kappa)"""
        return self.sigma**2 * (1 - np.exp(-2 * self.kappa * t)) / (2 * self.kappa)


## 3. Monte Carlo Engine

The `MonteCarloEngine` wraps any `StochasticProcess` and provides:
- Multi-path simulation with antithetic variates
- Terminal distribution statistics (mean, std, skew, kurtosis)
- Risk metrics: VaR, CVaR (Expected Shortfall) at arbitrary confidence levels
- Convergence diagnostics vs. analytical benchmarks (GBM only)


In [ ]:
class MonteCarloEngine:
    """
    Generic Monte Carlo engine compatible with any StochasticProcess subclass.
    """
    def __init__(self, process: StochasticProcess, T: float, N: int, M: int):
        self.process = process
        self.T       = T
        self.N       = N
        self.M       = M
        self._paths  = None   # cached after run()

    def run(self, antithetic: bool = True) -> np.ndarray:
        """Simulate paths and cache results. Returns (M, N+1) array."""
        self._paths = self.process.simulate(self.T, self.N, self.M, antithetic)
        return self._paths

    @property
    def paths(self) -> np.ndarray:
        if self._paths is None:
            raise RuntimeError("Call run() first.")
        return self._paths

    @property
    def terminal_prices(self) -> np.ndarray:
        """Final prices S(T) across all paths."""
        return self.paths[:, -1]

    def statistics(self) -> dict:
        """Descriptive statistics of the terminal distribution."""
        from scipy.stats import skew, kurtosis
        S_T = self.terminal_prices
        return {
            "mean":     S_T.mean(),
            "std":      S_T.std(),
            "skewness": skew(S_T),
            "kurtosis": kurtosis(S_T),   # excess kurtosis
            "min":      S_T.min(),
            "max":      S_T.max(),
        }

    def var(self, confidence: float = 0.95) -> float:
        """
        Historical VaR: loss not exceeded with probability `confidence`.
        Expressed as a positive number (loss convention).
        VaR_alpha = S0 - quantile_{1-alpha}(S_T)
        """
        q = np.percentile(self.terminal_prices, (1 - confidence) * 100)
        return self.process.S0 - q

    def cvar(self, confidence: float = 0.95) -> float:
        """
        Conditional VaR (Expected Shortfall): expected loss given loss > VaR.
        CVaR_alpha = S0 - E[S_T | S_T < quantile_{1-alpha}(S_T)]
        """
        q = np.percentile(self.terminal_prices, (1 - confidence) * 100)
        tail = self.terminal_prices[self.terminal_prices <= q]
        return self.process.S0 - tail.mean() if len(tail) > 0 else np.nan

    def convergence_study(self, M_values: list, confidence: float = 0.95) -> dict:
        """
        Run simulations for increasing M and track mean/VaR convergence.
        Useful for validating that M is large enough.
        """
        means, vars_ = [], []
        for m in M_values:
            eng = MonteCarloEngine(self.process, self.T, self.N, m)
            eng.run()
            means.append(eng.terminal_prices.mean())
            vars_.append(eng.var(confidence))
        return {"M": M_values, "mean": means, f"VaR_{confidence}": vars_}


## 4. Visualisation Utilities

In [ ]:
def plot_paths(paths: np.ndarray, T: float, title: str,
              n_display: int = 100, color: str = 'steelblue') -> None:
    """Plot a subset of simulated paths with mean and confidence band."""
    N   = paths.shape[1] - 1
    t   = np.linspace(0, T, N + 1)
    mean_path = paths.mean(axis=0)
    p5, p95   = np.percentile(paths, [5, 95], axis=0)

    fig, ax = plt.subplots(figsize=(11, 5))
    ax.plot(t, paths[:n_display].T, alpha=0.08, color=color, lw=0.8)
    ax.plot(t, mean_path, color='crimson',   lw=2,   label='Mean path')
    ax.fill_between(t, p5, p95, alpha=0.18, color=color, label='90% CI')
    ax.set_title(title, fontsize=13)
    ax.set_xlabel('Time (years)')
    ax.set_ylabel('Price / Level')
    ax.legend()
    plt.tight_layout()
    plt.show()


def plot_terminal_distribution(engine: MonteCarloEngine, title: str,
                               analytical_gbm: GeometricBrownianMotion = None) -> None:
    """
    Plot the terminal price histogram with VaR/CVaR markers.
    Overlays the analytical GBM log-normal PDF if provided.
    """
    S_T  = engine.terminal_prices
    var  = engine.var(0.95)
    cvar = engine.cvar(0.95)
    S0   = engine.process.S0

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(S_T, bins=80, density=True, color='steelblue', alpha=0.65, label='MC distribution')

    # Analytical overlay for GBM
    if analytical_gbm is not None:
        T   = engine.T
        mu_ln  = np.log(S0) + (analytical_gbm.mu - 0.5 * analytical_gbm.sigma**2) * T
        sig_ln = analytical_gbm.sigma * np.sqrt(T)
        x = np.linspace(S_T.min(), S_T.max(), 400)
        pdf = lognorm.pdf(x, s=sig_ln, scale=np.exp(mu_ln))
        ax.plot(x, pdf, 'r-', lw=2, label='Analytical log-normal PDF')

    # VaR / CVaR markers
    ax.axvline(S0 - var,  color='orange', lw=2, linestyle='--', label=f'VaR 95% = {var:.2f}')
    ax.axvline(S0 - cvar, color='red',    lw=2, linestyle=':',  label=f'CVaR 95% = {cvar:.2f}')

    ax.set_title(title, fontsize=13)
    ax.set_xlabel('Terminal Price S(T)')
    ax.set_ylabel('Density')
    ax.legend()
    plt.tight_layout()
    plt.show()

    stats = engine.statistics()
    print(f"  Mean:      {stats['mean']:.4f}   |  Std:      {stats['std']:.4f}")
    print(f"  Skewness:  {stats['skewness']:.4f}   |  Ex. Kurt: {stats['kurtosis']:.4f}")
    print(f"  VaR 95%:   {var:.4f}   |  CVaR 95%: {cvar:.4f}")


def plot_convergence(conv: dict, analytical_mean: float = None) -> None:
    """Plot mean and VaR convergence as M increases."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    key_var = [k for k in conv if k.startswith('VaR')][0]

    axes[0].plot(conv['M'], conv['mean'], 'o-', color='steelblue', label='MC mean')
    if analytical_mean is not None:
        axes[0].axhline(analytical_mean, color='red', linestyle='--', label='Analytical mean')
    axes[0].set_xlabel('Number of paths M')
    axes[0].set_ylabel('E[S(T)]')
    axes[0].set_title('Mean Convergence')
    axes[0].legend()

    axes[1].plot(conv['M'], conv[key_var], 's-', color='darkorange', label=key_var)
    axes[1].set_xlabel('Number of paths M')
    axes[1].set_ylabel('VaR')
    axes[1].set_title('VaR 95% Convergence')
    axes[1].legend()

    plt.tight_layout()
    plt.show()


## 5. Numerical Applications

### 5.1 — Geometric Brownian Motion

In [ ]:
# Parameters
gbm = GeometricBrownianMotion(S0=100, mu=0.08, sigma=0.20)
T, N, M = 1.0, 252, 10_000

engine_gbm = MonteCarloEngine(gbm, T, N, M)
engine_gbm.run(antithetic=True)

plot_paths(engine_gbm.paths, T, 'GBM — 10,000 Simulated Paths (1 Year)', n_display=150)


In [ ]:
# Terminal distribution vs. analytical log-normal
plot_terminal_distribution(engine_gbm, 'GBM — Terminal Distribution vs. Analytical PDF',
                           analytical_gbm=gbm)

# Validation against analytical moments
print("\nAnalytical vs. Monte Carlo validation:")
print(f"  Analytical E[S(T)]:  {gbm.analytical_mean(T):.4f}")
print(f"  MC E[S(T)]:          {engine_gbm.terminal_prices.mean():.4f}")
print(f"  Analytical Std[S(T)]: {gbm.analytical_std(T):.4f}")
print(f"  MC Std[S(T)]:         {engine_gbm.terminal_prices.std():.4f}")
print(f"  Analytical VaR 95%:  {gbm.analytical_var(T, 0.95):.4f}")
print(f"  MC VaR 95%:          {engine_gbm.var(0.95):.4f}")


In [ ]:
# Convergence study
M_vals = [500, 1000, 2000, 5000, 10000, 20000, 50000]
conv = engine_gbm.convergence_study(M_vals)
plot_convergence(conv, analytical_mean=gbm.analytical_mean(T))


### 5.2 — Merton Jump Diffusion

In [ ]:
# Jump parameters: moderate downward jumps, ~3 per year
mjd = MertonJumpDiffusion(S0=100, mu=0.08, sigma=0.18,
                          jump_mu=-0.10, jump_sigma=0.12, jump_lambda=3.0)

engine_mjd = MonteCarloEngine(mjd, T, N, M)
engine_mjd.run(antithetic=True)

plot_paths(engine_mjd.paths, T, 'Merton Jump Diffusion — 10,000 Paths', color='darkorange')


In [ ]:
plot_terminal_distribution(engine_mjd, 'Jump Diffusion — Terminal Distribution')

# Compare GBM vs. Jump Diffusion risk metrics
print("\nRisk Metrics Comparison (95% confidence, T=1Y):")
print(f"{'Metric':<20} {'GBM':>12} {'Jump Diffusion':>16}")
print("-" * 50)
for conf in [0.95, 0.99]:
    v_gbm = engine_gbm.var(conf)
    v_mjd = engine_mjd.var(conf)
    c_gbm = engine_gbm.cvar(conf)
    c_mjd = engine_mjd.cvar(conf)
    print(f"VaR  {conf*100:.0f}%:         {v_gbm:>12.4f} {v_mjd:>16.4f}")
    print(f"CVaR {conf*100:.0f}%:         {c_gbm:>12.4f} {c_mjd:>16.4f}")


### 5.3 — Ornstein-Uhlenbeck (Mean-Reverting)

In [ ]:
# Short-rate model: kappa=2 (half-life ~4 months), theta=3%, sigma=1%
ou = OrnsteinUhlenbeck(X0=0.05, kappa=2.0, theta=0.03, sigma=0.01)
engine_ou = MonteCarloEngine(ou, T=5.0, N=252*5, M=5_000)
engine_ou.run(antithetic=True)

plot_paths(engine_ou.paths, T=5.0, 'Ornstein-Uhlenbeck — Interest Rate Simulation (5Y)',
           color='seagreen', n_display=100)

# Validate against analytical mean and variance
t_vals = np.array([0.5, 1, 2, 5])
print("\nOU Process — Analytical vs. MC moments at selected horizons:")
print(f"{'t':>5} {'E[X] analytic':>16} {'E[X] MC':>12} {'Var analytic':>16} {'Var MC':>10}")
N_total = engine_ou.paths.shape[1] - 1
for t in t_vals:
    idx = int(t / 5.0 * N_total)
    mc_mean = engine_ou.paths[:, idx].mean()
    mc_var  = engine_ou.paths[:, idx].var()
    print(f"{t:>5.1f} {ou.analytical_mean(t):>16.6f} {mc_mean:>12.6f} "
          f"{ou.analytical_var_process(t):>16.8f} {mc_var:>10.8f}")


### 5.4 — Interactive Simulation Dashboard

In [ ]:
def interactive_simulation(
    model       = 'GBM',
    mu          = 0.08,
    sigma       = 0.20,
    S0          = 100.0,
    jump_mu     = -0.10,
    jump_sigma  = 0.12,
    jump_lambda = 3.0,
    kappa       = 2.0,
    theta       = 0.03,
    T           = 1.0,
    N           = 252,
    M           = 5000,
):
    np.random.seed(0)
    if model == 'GBM':
        proc = GeometricBrownianMotion(S0, mu, sigma)
        title = 'GBM'
    elif model == 'Jump Diffusion':
        proc = MertonJumpDiffusion(S0, mu, sigma, jump_mu, jump_sigma, jump_lambda)
        title = 'Merton Jump Diffusion'
    else:
        proc = OrnsteinUhlenbeck(S0, kappa, theta, sigma)
        title = 'Ornstein-Uhlenbeck'

    eng = MonteCarloEngine(proc, T, N, M)
    eng.run(antithetic=True)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    t_arr = np.linspace(0, T, N + 1)
    axes[0].plot(t_arr, eng.paths[:100].T, alpha=0.1, color='steelblue', lw=0.8)
    axes[0].plot(t_arr, eng.paths.mean(axis=0), color='red', lw=2, label='Mean')
    axes[0].set_title(f'{title} — Paths (T={T}Y, M={M})')
    axes[0].set_xlabel('Time'); axes[0].legend()

    S_T = eng.terminal_prices
    axes[1].hist(S_T, bins=60, density=True, color='steelblue', alpha=0.7)
    axes[1].axvline(S_T.mean(), color='red', lw=2, label=f'Mean={S_T.mean():.1f}')
    axes[1].axvline(np.percentile(S_T, 5), color='orange', lw=2, linestyle='--',
                    label=f'5th pct={np.percentile(S_T,5):.1f}')
    axes[1].set_title('Terminal Distribution')
    axes[1].set_xlabel('S(T)'); axes[1].legend()
    plt.tight_layout(); plt.show()

    stats = eng.statistics()
    print(f"Mean={stats['mean']:.3f}  Std={stats['std']:.3f}  "
          f"Skew={stats['skewness']:.3f}  ExKurt={stats['kurtosis']:.3f}  "
          f"VaR95={eng.var():.3f}  CVaR95={eng.cvar():.3f}")

interact(
    interactive_simulation,
    model       = Dropdown(options=['GBM', 'Jump Diffusion', 'Ornstein-Uhlenbeck'], description='Model'),
    mu          = FloatSlider(value=0.08, min=-0.3, max=0.5,  step=0.01, description='Drift μ'),
    sigma       = FloatSlider(value=0.20, min=0.01, max=0.8,  step=0.01, description='Vol σ'),
    S0          = FloatSlider(value=100,  min=10,   max=500,  step=10,   description='S0'),
    jump_mu     = FloatSlider(value=-0.10,min=-1.0, max=0.5,  step=0.01, description='Jump μ_J'),
    jump_sigma  = FloatSlider(value=0.12, min=0.01, max=0.5,  step=0.01, description='Jump σ_J'),
    jump_lambda = FloatSlider(value=3.0,  min=0.1,  max=20,   step=0.5,  description='Jump λ'),
    kappa       = FloatSlider(value=2.0,  min=0.1,  max=10,   step=0.1,  description='OU κ'),
    theta       = FloatSlider(value=0.03, min=-0.1, max=0.2,  step=0.005,description='OU θ'),
    T           = FloatSlider(value=1.0,  min=0.1,  max=10,   step=0.1,  description='Horizon T'),
    N           = IntSlider(  value=252,  min=50,   max=1000, step=50,   description='Steps N'),
    M           = IntSlider(  value=5000, min=500,  max=20000,step=500,  description='Paths M'),
);
